# Image Captioning with Generative AI

**Objective:** Understand how Generative AI can analyze an image and generate a meaningful text description.

**Tech Stack:** Python, Hugging Face Transformers, PyTorch, PIL, Jupyter Notebook

In this notebook we will:
1. Load a pretrained image captioning model (Salesforce's **BLIP** model from Hugging Face).
2. Select / upload images.
3. Pass each image to the model.
4. Generate a caption for each image.
5. Try the model with 3 different images.
6. Display each image along with its generated caption.

## 1. Install & Import Dependencies

Run the cell below once to make sure all required packages are installed. If you're running this in an environment that already has these packages (e.g. Google Colab often does), you can skip or comment out the `pip install` line.

In [ ]:
# Uncomment the line below if the packages aren't already installed in your environment
# !pip install transformers torch torchvision pillow requests --quiet

In [ ]:
import torch
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

from transformers import BlipProcessor, BlipForConditionalGeneration

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

## 2. Load a Pretrained Image Captioning Model

We use **`Salesforce/blip-image-captioning-base`**, a pretrained vision-language model hosted on Hugging Face. 

- The **processor** prepares the raw image (resizing, normalizing, converting to tensors) so it matches what the model expects.
- The **model** is a transformer that has learned, from millions of image-caption pairs, how to map visual features to natural language descriptions.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)

print(f"Model '{model_name}' loaded successfully on {device}.")

## 3. Helper Functions

Below are two small helper functions:

- `load_image(source)`: loads an image either from a local file path or a URL.
- `generate_caption(image)`: passes the image through the processor + model to produce a text caption.

In [ ]:
def load_image(source: str) -> Image.Image:
    """Load an image from a local file path or a URL and return a PIL Image (RGB)."""
    if source.startswith("http://") or source.startswith("https://"):
        response = requests.get(source)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(source).convert("RGB")
    return image


def generate_caption(image: Image.Image) -> str:
    """Generate a text caption for a given PIL image using the BLIP model."""
    inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=50)
    caption = processor.decode(output_ids[0], skip_special_tokens=True)
    return caption


def show_image_with_caption(image: Image.Image, caption: str, title: str = ""):
    """Display an image alongside its generated caption."""
    plt.figure(figsize=(6, 6))
    plt.imshow(image)
    plt.axis("off")
    plt.title(f"{title}\nCaption: {caption}", fontsize=11, wrap=True)
    plt.show()

## 4. Try the Model on 3 Different Images

You have two options for supplying images:

### Option A — Use sample image URLs (no upload needed)
Run the cell below to caption 3 sample images directly from the web.

### Option B — Upload your own images
If working in Jupyter locally, place your image files (e.g. `dog.jpg`, `beach.jpg`, `city.jpg`) in the same folder as this notebook and update the `image_sources` list with their file paths. In Google Colab, use the file upload widget provided further below.

In [ ]:
# Option A: Sample images (feel free to replace these URLs with your own image paths or URLs)
image_sources = [
    "https://images.unsplash.com/photo-1552053831-71594a27632d",  # dog in a park
    "https://images.unsplash.com/photo-1507525428034-b723cf961d3e",  # beach scene
    "https://images.unsplash.com/photo-1519501025264-65ba15a82390",  # city street
]

print(f"Number of images to process: {len(image_sources)}")

In [ ]:
# --- Option B (optional): Upload images if running in Google Colab ---
# Uncomment and run this cell instead of Option A if you want to upload your own images.

# from google.colab import files
# uploaded = files.upload()
# image_sources = list(uploaded.keys())  # uses the uploaded file names as paths

In [ ]:
# Loop through the 3 images: load each, generate a caption, and display the result
results = []

for i, source in enumerate(image_sources, start=1):
    image = load_image(source)
    caption = generate_caption(image)
    results.append((source, caption))
    show_image_with_caption(image, caption, title=f"Image {i}")
    print(f"Image {i}: {source}\nGenerated Caption: {caption}\n{'-'*60}")

## 5. Summary of Results

A quick recap table of all the captions generated in this run.

In [ ]:
print(f"{'Image #':<10}{'Caption'}")
print("-" * 60)
for i, (source, caption) in enumerate(results, start=1):
    print(f"{i:<10}{caption}")

## 6. How It Works (Conceptual Overview)

1. **Vision Encoder**: The image is split into patches and passed through a vision transformer (or CNN, depending on the architecture) to produce a set of visual feature embeddings that capture objects, scenes, and their relationships.
2. **Text Decoder**: A transformer-based language decoder takes those visual embeddings as context and generates text one token at a time, conditioning each new word on the image features and the words generated so far.
3. **Pretraining**: The model (BLIP) was pretrained on large datasets of image-caption pairs scraped from the web, learning a shared representation space between vision and language.
4. **Generation**: At inference time, we simply feed in a new image; the model uses beam search / greedy decoding (via `model.generate()`) to produce the most likely caption given what it learned during training.

This is a form of **multimodal Generative AI**: it doesn't retrieve a pre-written caption — it generates a novel sequence of text conditioned on visual input, similar in spirit to how a language model generates text conditioned on a prompt.

### Try it yourself
- Replace the image URLs with your own images (upload or link).
- Try a different pretrained model, e.g. `"nlpconnect/vit-gpt2-image-captioning"` or `"Salesforce/blip-image-captioning-large"`, and compare the captions.
- Experiment with `max_new_tokens` or decoding strategies (e.g. `num_beams=5` in `model.generate()`) to see how caption quality/length changes.